# Aula 2 - Context Engineering

## Versão 2.0 do projeto: um assistente que seleciona contexto

Nesta aula, a aplicação vai selecionar documentos locais antes de chamar uma LLM. Precisamos responder: "qual informação entra na requisição para responder esta pergunta?"

Ao final, você deve conseguir:

- separar instruções, pergunta, histórico e dados externos;
- selecionar documentos por uma regra simples de relevância;
- limitar o tamanho do contexto;
- montar uma requisição que identifica suas fontes;
- comparar uma resposta sem contexto com uma resposta contextualizada.

## 1. Configuração do ambiente

Instale as dependências pelo terminal, seguindo o `README.md` desta pasta. Copie `env.example` para `.env` e preencha `GROQ_API_KEY`. Nunca coloque a chave diretamente neste arquivo nem faça commit do `.env`.

In [9]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise RuntimeError(
        "GROQ_API_KEY não encontrada. Copie env.example para .env e informe sua chave."
    )

client = OpenAI(
    api_key=api_key,
    base_url=os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1"),
)

MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")
print(f"Modelo configurado: {MODEL}")

Modelo configurado: openai/gpt-oss-120b


## 2. Crie uma base local de documentos

Cada documento tem um identificador, um título, um conteúdo e termos que ajudam a explicar por que ele foi selecionado. Em um sistema real, essa base poderia vir de arquivos, banco de dados ou uma API.

In [10]:
knowledge_base = [
    {
        "id": "doc001",
        "title": "Introdução à Programação em Python",
        "keywords": ["python", "programação", "variáveis", "funções", "laços", "listas"],
        "content": (
            "Python é uma linguagem de programação de alto nível, interpretada e de "
            "tipagem dinâmica, conhecida pela sintaxe simples e legível. Variáveis "
            "armazenam valores sem a necessidade de declarar o tipo. Estruturas de "
            "controle como if, for e while permitem tomar decisões e repetir ações. "
            "Funções são definidas com a palavra-chave def e ajudam a reutilizar código. "
            "Listas, tuplas, dicionários e conjuntos são as principais estruturas de "
            "dados nativas da linguagem."
        ),
    },
    {
        "id": "doc002",
        "title": "Fundamentos de Banco de Dados",
        "keywords": ["banco de dados", "sql", "tabelas", "chave primária", "consultas", "normalização"],
        "content": (
            "Um banco de dados relacional organiza informações em tabelas compostas por "
            "linhas e colunas. Cada tabela possui uma chave primária que identifica "
            "unicamente cada registro, e chaves estrangeiras estabelecem relacionamentos "
            "entre tabelas. A linguagem SQL é usada para consultar e manipular dados, com "
            "comandos como SELECT, INSERT, UPDATE e DELETE. A normalização é o processo "
            "de organizar os dados para reduzir redundâncias e evitar inconsistências."
        ),
    },
    {
        "id": "doc003",
        "title": "Estruturas de Dados e Algoritmos",
        "keywords": ["algoritmos", "estruturas de dados", "pilha", "fila", "ordenação", "complexidade"],
        "content": (
            "Estruturas de dados são formas de organizar informações na memória para "
            "facilitar o acesso e a modificação. Pilhas seguem o princípio LIFO (último "
            "a entrar, primeiro a sair), enquanto filas seguem o princípio FIFO (primeiro "
            "a entrar, primeiro a sair). Algoritmos de ordenação, como Bubble Sort, "
            "Merge Sort e Quick Sort, organizam elementos em uma sequência. A notação "
            "Big O é usada para descrever a complexidade de tempo e espaço de um algoritmo."
        ),
    },
    {
        "id": "doc004",
        "title": "Introdução à Inteligência Artificial",
        "keywords": ["inteligência artificial", "machine learning", "redes neurais", "dados", "modelos"],
        "content": (
            "Inteligência Artificial é a área da computação que busca criar sistemas "
            "capazes de executar tarefas que normalmente exigem inteligência humana. "
            "O aprendizado de máquina (machine learning) é um subcampo da IA em que "
            "modelos aprendem padrões a partir de dados. Pode ser supervisionado, "
            "não supervisionado ou por reforço. Redes neurais artificiais, inspiradas "
            "no cérebro humano, são a base do aprendizado profundo (deep learning)."
        ),
    },
    {
        "id": "doc005",
        "title": "Redes de Computadores",
        "keywords": ["redes", "internet", "protocolos", "tcp/ip", "endereço ip", "http"],
        "content": (
            "Redes de computadores permitem a comunicação e o compartilhamento de "
            "recursos entre dispositivos. O modelo TCP/IP organiza essa comunicação em "
            "camadas: aplicação, transporte, internet e acesso à rede. Cada dispositivo "
            "conectado recebe um endereço IP que o identifica na rede. Protocolos como "
            "HTTP, HTTPS, FTP e DNS definem regras para a troca de informações, sendo o "
            "HTTP a base da navegação na web."
        ),
    },
    {
        "id": "doc006",
        "title": "Engenharia de Software",
        "keywords": ["engenharia de software", "requisitos", "metodologias ágeis", "scrum", "testes"],
        "content": (
            "Engenharia de Software é a disciplina que aplica princípios de engenharia "
            "ao desenvolvimento de sistemas. Envolve etapas como levantamento de "
            "requisitos, projeto, implementação, testes e manutenção. Metodologias "
            "ágeis, como Scrum e Kanban, priorizam entregas incrementais e colaboração "
            "com o cliente. Testes de software, como testes unitários e de integração, "
            "garantem a qualidade e reduzem a ocorrência de falhas."
        ),
    },
]


## 3. Selecione o contexto

Vamos usar uma estratégia transparente: normalizar os termos da pergunta e contar quantos deles aparecem em cada documento. Isso é suficiente para enxergar a etapa de seleção, mas não entende bem sinônimos.

In [11]:
STOPWORDS = {
    "a", "o", "e", "de", "da", "do", "das", "dos", "em",
    "um", "uma", "para", "por", "com", "que", "na", "no",
    "as", "os", "qual", "quais", "como", "é", "são",
}

# Normalizamos o prompt do usuário, simulando a vetorização.
def normalize_terms(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    return {term for term in text.split() if len(term) > 2 and term not in STOPWORDS}

# Ajudamos o agente a retornar com o contexto que mais possui os termos do prompt, simulando o retrieval.
def select_context(query, documents, max_docs=3, max_chars=1800):
    query_terms = normalize_terms(query)
    ranked = []

    for document in documents:
        searchable = " ".join([
            document["title"],
            document["content"],
            " ".join(document.get("keywords", [])),
        ])
        score = len(query_terms & normalize_terms(searchable))
        if score > 0:
            ranked.append({**document, "score": score})

    ranked.sort(key=lambda document: (-document["score"], document["id"]))

    selected = []
    used_chars = 0
    for document in ranked:
        cost = len(document["title"]) + len(document["content"])
        if selected and used_chars + cost > max_chars:
            continue
        selected.append(document)
        used_chars += cost
        if len(selected) >= max_docs:
            break

    return selected

## 4. Formate o contexto com fontes

O modelo precisa distinguir os dados externos da pergunta. Também queremos preservar as fontes para que a resposta consiga apontar de onde veio a informação.

In [12]:
# O bloco contém ids de fonte para facilitar inspeção e atribuição. O modelo recebe dados externos claramente separados da pergunta.
def build_context(selected_documents):
    if not selected_documents:
        return "Nenhum documento relevante foi selecionado."

    blocks = ["<contexto>"]
    for document in selected_documents:
        blocks.append(
            f"[fonte: {document['id']} | relevância: {document['score']}]\n"
            f"{document['title']}\n{document['content']}"
        )
    blocks.append("</contexto>")
    return "\n\n".join(blocks)

## 5. Veja o que entrou no contexto

Antes de chamar a LLM, inspecione a decisão da aplicação. Context Engineering também é observabilidade: precisamos conseguir explicar qual informação foi enviada.

In [15]:
import re

query = "Como aprender redes?"
selected = select_context(query, knowledge_base)
context = build_context(selected)

print(context)
print(f"\nDocumentos selecionados: {len(selected)}")
print(f"Caracteres de contexto: {len(context)}")

<contexto>

[fonte: doc004 | relevância: 1]
Introdução à Inteligência Artificial
Inteligência Artificial é a área da computação que busca criar sistemas capazes de executar tarefas que normalmente exigem inteligência humana. O aprendizado de máquina (machine learning) é um subcampo da IA em que modelos aprendem padrões a partir de dados. Pode ser supervisionado, não supervisionado ou por reforço. Redes neurais artificiais, inspiradas no cérebro humano, são a base do aprendizado profundo (deep learning).

[fonte: doc005 | relevância: 1]
Redes de Computadores
Redes de computadores permitem a comunicação e o compartilhamento de recursos entre dispositivos. O modelo TCP/IP organiza essa comunicação em camadas: aplicação, transporte, internet e acesso à rede. Cada dispositivo conectado recebe um endereço IP que o identifica na rede. Protocolos como HTTP, HTTPS, FTP e DNS definem regras para a troca de informações, sendo o HTTP a base da navegação na web.

</contexto>

Documentos selecionado

## 6. Monte a requisição com contexto

A instrução do sistema define como tratar as fontes. A mensagem do usuário contém o contexto selecionado e a pergunta atual. A aplicação continua responsável por montar tudo antes da chamada.

In [23]:
# TODO: crie `messages_com_contexto` com uma mensagem system e uma user.
# Instrua o assistente a usar apenas o contexto fornecido, citar os ids
# das fontes quando possível e dizer quando o contexto não for suficiente.
# Envie a lista para o modelo e salve a resposta em `response_com_contexto`.

messages_com_contexto = [
    {
        "role": "system",
        "content": (
            "Voce sera o mentor para guiar os estudos, tente sempre explicar de forma clara e detalhada."
            "Utilize apenas as informacoes dentro de <contexto>. "
            "Lembre-se de citar os ids das fontes quando se basear nelas. "
            "Em caso do contexto nao ser suficiente, seja explicito em relacao a isso."
        )
    },
    {
        "role": "user",
        "content": f"{context}\n\nPergunta: {query}",
    },
]

response_com_contexto = client.chat.completions.create(
    model=MODEL,
    messages=messages_com_contexto,
)

In [24]:
# TODO: mostre o texto de `response_com_contexto` e os ids das fontes selecionadas.

print(response_com_contexto.choices[0].message.content)
print(f"\nFontes enviadas: {[document['id'] for document in selected]}")

Claro! Vou orientar você passo a passo sobre como iniciar os estudos em **redes de computadores**, usando as informações que temos no contexto e indicando onde o material disponível pode ser útil.

---

## 1. Entenda o que são redes de computadores
- **Definição básica**: redes permitem a comunicação e o compartilhamento de recursos entre dispositivos [fonte: **doc005**].  
- **Objetivo inicial**: saber que cada dispositivo conectado recebe um **endereço IP** que o identifica na rede [fonte: **doc005**].

---

## 2. Comece pelo modelo TCP/IP
O modelo TCP/IP é a espinha dorsal da maioria das redes atuais. Ele está dividido em **quatro camadas**:

| Camada | Função principal | Exemplos de protocolos |
|--------|------------------|------------------------|
| **Aplicação** | Interação direta com o usuário (ex.: navegadores, e‑mail) | HTTP, HTTPS, FTP, DNS |
| **Transporte** | Controle de fluxo e entrega confiável dos dados | TCP, UDP |
| **Internet** | Endereçamento e roteamento entre rede

In [ ]:
# Exemplo de instruções para o assistente:
# messages_com_contexto = [
#     {
#         "role": "system",
#         "content": (
#             "Você é um assistente de estudos. Use apenas as informações dentro de <contexto>. "
#             "Cite os ids das fontes quando fizer uma afirmação baseada nelas. "
#             "Se o contexto não for suficiente, diga isso claramente."
#         ),
#     },
#     {
#         "role": "user",
#         "content": f"{context}\n\nPergunta: {query}",
#     },
# ]

## 7. Compare sem contexto e com contexto

Escolha uma pergunta que dependa de uma informação da `knowledge_base`. Faça duas chamadas independentes. A primeira recebe apenas a pergunta; a segunda recebe o bloco selecionado. Compare especificidade, fontes e grau de confiança.

In [25]:
prompt_2 = "Quais protocolos de rede são citados e qual deles é a base da navegação na web?"

response_sem_contexto = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": prompt_2},
    ],
)

selected_2 = select_context(prompt_2, knowledge_base)
context_2 = build_context(selected_2)

messages_com_contexto_2 = [
    {
        "role": "system",
        "content": (
            "Você é um assistente de estudos. Use apenas as informações dentro de <contexto>. "
            "Cite os ids das fontes quando fizer uma afirmação baseada nelas. "
            "Se o contexto não for suficiente, diga isso claramente."
        ),
    },
    {
        "role": "user",
        "content": f"{context_2}\n\nPergunta: {prompt_2}",
    },
]

response_com_contexto_2 = client.chat.completions.create(
    model=MODEL,
    messages=messages_com_contexto_2,
)

print("=" * 80)
print("SEM CONTEXTO")
print("=" * 80)
print(response_sem_contexto.choices[0].message.content)

print("\n" + "=" * 80)
print("COM CONTEXTO")
print("=" * 80)
print(response_com_contexto_2.choices[0].message.content)
print(f"\nFontes enviadas: {[document['id'] for document in selected_2]}")

SEM CONTEXTO
Os protocolos de rede mais frequentemente citados quando se fala em comunicação na Internet são:

| Categoria | Protocolos citados | Função principal |
|-----------|--------------------|------------------|
| **Camada de transporte** | **TCP** (Transmission Control Protocol) – orientado à conexão, garante a entrega correta e ordenada dos pacotes.<br>UDP (User Datagram Protocol) – sem conexão, usado quando a velocidade é mais importante que a confiabilidade (ex.: streaming, DNS). | Controle de fluxo, retransmissão, ordenação dos dados. |
| **Camada de rede** | **IP** (Internet Protocol – IPv4/IPv6) – endereçamento e roteamento dos pacotes entre dispositivos. | Endereçamento lógico (endereços IP) e roteamento entre redes. |
| **Camada de aplicação** | **HTTP** (HyperText Transfer Protocol) – troca de documentos hypertextuais (HTML, CSS, JS).<br>**HTTPS** (HTTP Secure) – versão criptografada do HTTP (sobre TLS/SSL).<br>**FTP** (File Transfer Protocol) – transferência de arquiv

## 8. Experimente o orçamento de contexto

Altere `max_chars` na função de seleção. Observe o que desaparece quando o orçamento diminui e por que a aplicação precisa escolher uma política explícita.

In [26]:
for budget in [500, 1800]:
    selected_for_budget = select_context(
        query, knowledge_base, max_docs=3, max_chars=budget
    )
    context_for_budget = build_context(selected_for_budget)
    print(f"orçamento={budget}: ids={[document['id'] for document in selected_for_budget]} | caracteres={len(context_for_budget)}")

orçamento=500: ids=['doc004'] | caracteres=521
orçamento=1800: ids=['doc004', 'doc005'] | caracteres=976


## 9. Checkpoint da V2

A V2 está pronta quando:

- a base de documentos existe;
- a seleção ocorre antes da chamada;
- o limite de documentos e caracteres é explícito;
- o contexto preserva as fontes;
- você comparou respostas sem contexto e com contexto;
- a chave continua no `.env`;
- as alterações foram salvas em um commit separado.
